# State and Molecule Parent Bug

This notebook illustrates parent-link bugs found in SCOPE 0.9.5. The main symptom was detected in an optimized State of the AJAXEP System: Metal.get_coord_sphere() returned atoms and coordinates from the original source Molecule instead of the Molecule stored in the optimized State.

The first two parts use small artificial systems so the tests do not depend on external data. The third part reproduces the original AJAXEP case when its System file is available. No System is saved or modified.

In [1]:
import os
from copy import deepcopy

import numpy as np
import scope

from scope.classes_cell import Cell
from scope.classes_specie import Molecule
from scope.classes_system import System
from scope.reconstruct import cart2frac

## Part 1. Molecule obtained from a State

Here we create a simple FeN6 Molecule. The optimized State has slightly shorter Fe-N distances than the source, making it easy to identify which coordinates are returned by get_coord_sphere().

In [2]:
labels = ['Fe'] + ['N'] * 6
source_coord = np.array([
    [ 0.0,  0.0,  0.0],
    [ 2.0,  0.0,  0.0],
    [-2.0,  0.0,  0.0],
    [ 0.0,  2.0,  0.0],
    [ 0.0, -2.0,  0.0],
    [ 0.0,  0.0,  2.0],
    [ 0.0,  0.0, -2.0],
])

optimized_coord = source_coord.copy()
optimized_coord[1:] *= 0.98

source_mol = Molecule(labels, source_coord.tolist())
source_mol.set_atoms(create_adjacencies=True)
source_mol.split_complex()

sys = System('state_parent_test')
sys.add_source('complex', source_mol)

optimized_state = source_mol.add_state('optimized')
optimized_state.set_geometry(labels, optimized_coord.tolist())

found, optimized_state = source_mol.find_state('optimized')
mol = optimized_state.molecules[0]
met = mol.metals[0]

print('State found:', found)
print(mol)

State found: True
-----------------------------------
------ SCOPE MOLECULE Object ------
-----------------------------------
 Version               = 1.0
 Type                  = specie
 Sub-Type              = molecule
 Number of Atoms       = 7
 Formula               = N6-Fe
 Charge                = 0
 Spin (alpha - beta)   = 0
 Number of Parents     = 1
 Has Adjacency Matrix  = YES
 Has Bonds             = NO

 Num of Ligands        = 6
   Ligand 0: N with 1 atoms.
   Ligand 1: N with 1 atoms.
   Ligand 2: N with 1 atoms.
   Ligand 3: N with 1 atoms.
   Ligand 4: N with 1 atoms.
   Ligand 5: N with 1 atoms.

 Num of Metals         = 1
   Metal 0: Fe with charge:0 and spin:0




### Parent relationships

The Molecule was created from optimized_state, so the State should be its parent. In turn, the Metal should identify mol as its parent Molecule.

In [3]:
state_parent = mol.get_parent('state', search_by='type')
metal_parent = met.get_parent('molecule')

print('State is parent of Molecule:', state_parent is optimized_state)
print('Optimized Molecule is parent of Metal:', metal_parent is mol)
print('Source Molecule is parent of Metal:', metal_parent is source_mol)

State is parent of Molecule: False
Optimized Molecule is parent of Metal: False
Source Molecule is parent of Metal: True


### Coordination sphere

For every atom in the coordination sphere, we compare the returned object and coordinates with those stored in the optimized Molecule. In SCOPE 0.9.5, these comparisons are False because the returned atoms belong to source_mol.

In [4]:
coord_sphere_idx = met.get_coord_sphere_idx()
coord_sphere = met.get_coord_sphere()

print('idx  label  Same object?  Same coordinates?  Source atom?')
print('-' * 62)
for idx, at in zip(coord_sphere_idx, coord_sphere):
    same_object = at is mol.atoms[idx]
    same_coordinates = np.allclose(at.coord, mol.coord[idx])
    source_atom = at is source_mol.atoms[idx]
    print(f'{idx:3}  {at.label:5}  {str(same_object):12}  {str(same_coordinates):17}  {source_atom}')

idx  label  Same object?  Same coordinates?  Source atom?
--------------------------------------------------------------
  1  N      False         False              True
  2  N      False         False              True
  3  N      False         False              True
  4  N      False         False              True
  5  N      False         False              True
  6  N      False         False              True


## Part 2. Molecules obtained from a periodic State

This Cell contains two separated C2 molecules. Its relaxed State has translated coordinates and different cell vectors. This case checks that State-derived Molecules use the State as parent and use the State cell vectors for fractional coordinates.

In [5]:
cell_labels = ['C', 'C', 'C', 'C']
cell_coord = np.array([
    [1.0, 1.0, 1.0],
    [2.4, 1.0, 1.0],
    [6.0, 6.0, 6.0],
    [7.4, 6.0, 6.0],
])

cell_vector = np.diag([10.0, 10.0, 10.0])
relaxed_cell_vector = np.diag([12.0, 11.0, 10.0])
relaxed_coord = cell_coord + np.array([0.1, 0.2, 0.3])

cell = Cell('periodic_test', cell_labels, cell_coord.tolist(), cell_vector=cell_vector)
cell.set_initial_state()

relaxed_state = cell.add_state('relaxed')
relaxed_state.set_geometry(cell_labels, relaxed_coord.tolist())
relaxed_state.set_cell(cell_vector=relaxed_cell_vector)

print('Number of Molecules:', len(relaxed_state.molecules))
for mol in relaxed_state.molecules:
    print(mol.formula, mol.coord)

Number of Molecules: 2
C2 [[6.1, 6.2, 6.3], [7.5, 6.2, 6.3]]
C2 [[1.1, 1.2, 1.3], [2.5, 1.2, 1.3]]


### State parents and indices

Each Molecule and Atom should point to relaxed_state. The Molecule indices in that parent should cover the complete State geometry.

In [6]:
all_state_indices = []

for idx, mol in enumerate(relaxed_state.molecules):
    parent = mol.get_parent('state', search_by='type')
    parent_indices = mol.get_parent_indices('state', search_by='type')
    atom_parents = [at.get_parent('state', search_by='type') is relaxed_state for at in mol.atoms]
    
    print(f'Molecule {idx}:')
    print('  State is parent:', parent is relaxed_state)
    print('  State indices:', parent_indices)
    print('  State is parent of all atoms:', all(atom_parents))
    print('  Source Cell is a direct parent:', any(p is cell for p in mol.parents))
    
    if parent_indices is not None:
        all_state_indices.extend(parent_indices)

print('Indices cover the State geometry:', sorted(all_state_indices) == list(range(relaxed_state.natoms)))

Molecule 0:
  State is parent: False
  State indices: None
  State is parent of all atoms: False
  Source Cell is a direct parent: True
Molecule 1:
  State is parent: False
  State indices: None
  State is parent of all atoms: False
  Source Cell is a direct parent: True
Indices cover the State geometry: False


### Fractional coordinates

The expected fractional coordinates are computed from the optimized Cartesian coordinates and the cell vectors stored in relaxed_state, not those in the source Cell.

In [7]:
for idx, mol in enumerate(relaxed_state.molecules):
    expected_frac_coord = cart2frac(mol.coord, relaxed_state.cell_vector)
    molecule_match = hasattr(mol, 'frac_coord') and np.allclose(mol.frac_coord, expected_frac_coord)
    atom_match = all(
        hasattr(at, 'frac_coord') and np.allclose(at.frac_coord, expected_frac_coord[jdx])
        for jdx, at in enumerate(mol.atoms)
    )
    
    print(f'Molecule {idx}:')
    print('  Molecule fractional coordinates match:', molecule_match)
    print('  Atom fractional coordinates match:', atom_match)

Molecule 0:
  Molecule fractional coordinates match: False
  Atom fractional coordinates match: False
Molecule 1:
  Molecule fractional coordinates match: False
  Atom fractional coordinates match: False


### Rebuilding State geometry from Molecules

State.set_geometry_from_molecules() should preserve the original atom order and coordinates. A copy is used so the original test State remains unchanged.

In [8]:
state_copy = deepcopy(relaxed_state)
expected_labels = state_copy.labels.copy()
expected_coord = np.array(state_copy.coord).copy()

state_copy.set_geometry_from_molecules()

print('Labels preserved:', state_copy.labels == expected_labels)
print('Coordinates preserved:', np.allclose(state_copy.coord, expected_coord))
print('Maximum coordinate difference:', np.max(np.abs(np.array(state_copy.coord) - expected_coord)))

Labels preserved: True
Coordinates preserved: False
Maximum coordinate difference: 5.0


## Part 3. AJAXEP

This section reproduces the original issue for the high-spin and low-spin B3LYP-optimized Molecules. get_molecules(overwrite=True) regenerates the Molecules in memory using the currently imported SCOPE code.

In [9]:
ajaxep_path = '/Users/sergivela/Documents/SCOPE/Database_SCO/Data/2-Systems/AJAXEP/AJAXEP.npy'

if os.path.isfile(ajaxep_path):
    import scope_sco
    debug_sys = scope.load_binary(ajaxep_path)
    
    found, source_hs = debug_sys.find_source('ref_hs_mol')
    found, state_hs = source_hs.find_state('b3lyp_opt')
    found, source_ls = debug_sys.find_source('ref_ls_mol')
    found, state_ls = source_ls.find_state('b3lyp_opt')
    
    state_hs.get_molecules(overwrite=True)
    state_ls.get_molecules(overwrite=True)
    
    mol_hs = state_hs.molecules[0]
    mol_ls = state_ls.molecules[0]
    print('AJAXEP loaded:', debug_sys.name)
else:
    print('AJAXEP System not found:', ajaxep_path)

AJAXEP loaded: AJAXEP


In [10]:
if os.path.isfile(ajaxep_path):
    for spin, state, mol in zip(['HS', 'LS'], [state_hs, state_ls], [mol_hs, mol_ls]):
        met = mol.metals[0]
        coord_sphere_idx = met.get_coord_sphere_idx()
        coord_sphere = met.get_coord_sphere()
        coord_difference = [
            np.linalg.norm(np.array(at.coord) - np.array(mol.coord[idx]))
            for idx, at in zip(coord_sphere_idx, coord_sphere)
        ]
        
        print(spin)
        print('  State is parent of Molecule:', mol.get_parent('state', search_by='type') is state)
        print('  Molecule is parent of Metal:', met.get_parent('molecule') is mol)
        print('  Coordination atoms belong to Molecule:', all(at is mol.atoms[idx] for idx, at in zip(coord_sphere_idx, coord_sphere)))
        print('  Maximum coordinate difference:', max(coord_difference))

HS
  State is parent of Molecule: False
  Molecule is parent of Metal: False
  Coordination atoms belong to Molecule: False
  Maximum coordinate difference: 0.9039488441318897
LS
  State is parent of Molecule: False
  Molecule is parent of Metal: False
  Coordination atoms belong to Molecule: False
  Maximum coordinate difference: 0.6367326969027122


## Regression assertions

Set test_fix = True after modifying the code. The assertions cover the synthetic examples and AJAXEP when its file is available. They are disabled by default because they are expected to fail in SCOPE 0.9.5.

In [11]:
test_fix = False

if test_fix:
    ## Molecular State
    test_mol = optimized_state.molecules[0]
    test_met = test_mol.metals[0]
    test_indices = test_met.get_coord_sphere_idx()
    test_sphere = test_met.get_coord_sphere()
    assert test_mol.get_parent('state', search_by='type') is optimized_state
    assert test_met.get_parent('molecule') is test_mol
    assert all(at is test_mol.atoms[idx] for idx, at in zip(test_indices, test_sphere))
    assert all(np.allclose(at.coord, test_mol.coord[idx]) for idx, at in zip(test_indices, test_sphere))
    
    ## Periodic State
    assert all(mol.get_parent('state', search_by='type') is relaxed_state for mol in relaxed_state.molecules)
    assert all(
        at.get_parent('state', search_by='type') is relaxed_state
        for mol in relaxed_state.molecules for at in mol.atoms
    )
    assert all(not any(parent is cell for parent in mol.parents) for mol in relaxed_state.molecules)
    assert sorted(all_state_indices) == list(range(relaxed_state.natoms))
    assert all(
        np.allclose(mol.frac_coord, cart2frac(mol.coord, relaxed_state.cell_vector))
        for mol in relaxed_state.molecules
    )
    assert state_copy.labels == expected_labels
    assert np.allclose(state_copy.coord, expected_coord)
    
    ## AJAXEP
    if os.path.isfile(ajaxep_path):
        for state, mol in zip([state_hs, state_ls], [mol_hs, mol_ls]):
            met = mol.metals[0]
            coord_sphere_idx = met.get_coord_sphere_idx()
            coord_sphere = met.get_coord_sphere()
            assert mol.get_parent('state', search_by='type') is state
            assert met.get_parent('molecule') is mol
            assert all(at is mol.atoms[idx] for idx, at in zip(coord_sphere_idx, coord_sphere))
            assert all(np.allclose(at.coord, mol.coord[idx]) for idx, at in zip(coord_sphere_idx, coord_sphere))
    
    print('All regression tests passed')